In [ ]:
# Locate the repository when Jupyter starts in a notebook subdirectory.
from pathlib import Path
import sys

_start = Path.cwd().resolve()
_repo = next((p for p in (_start, *_start.parents)
              if (p / "figure" / "paths.py").is_file()
              and (p / "run_cross_validation.py").is_file()), None)
if _repo is None:
    raise RuntimeError("Open this notebook inside the cloned sAge repository.")
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
from figure.paths import input_path, output_path, font_path


# figure-3-3-human-nonlinear-trajectory

Analyze human gene-expression trajectories.

Run Jupyter from the repository root. Required external data and results are listed in `figure/INPUTS.md`. Set `SAGE_FIGURE_INPUT_ROOT` and `SAGE_FIGURE_OUTPUT_ROOT` when using other directories. See figure/VALIDATION.md for the execution checks and their limits.


Pattern analysis across all selected genes.

In [ ]:
import os
import re
import math
import time
import textwrap
import logging

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as path_effects
from matplotlib.backends.backend_pdf import PdfPages
from scipy.interpolate import PchipInterpolator
from scipy.stats import pearsonr
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from statsmodels.nonparametric.smoothers_lowess import lowess
from tqdm import tqdm

try:
    import gseapy as gp
except ImportError:
    gp = None

# ==================== 1. Human paths and parameters ====================
INPUT_GENE_DIR = input_path("2-8.3-shanda/1-feature/1-human-guaidian-choose-gene/Gene_Lists")
DATA_DIR = input_path("2-8.3-shanda/1-data/GSE201333_RAW/2-human-tissue-filter-remove-AL-all-tissue")
GLOBAL_GENE_NAMES_FILE = input_path("2-8.3-shanda/1-data/GSE201333_RAW/gene_names_no_clones.txt")
OUTPUT_PLOTS_DIR = output_path("2-8.3-shanda/1-feature/1-figure/0-3-result-4-nonlinear/3-human-multi-gene-trajectory")

LABEL_TO_YEARS = {
    0: 22, 1: 33, 2: 37, 3: 38, 4: 40, 5: 42, 6: 46,
    7: 56, 8: 57, 9: 59, 10: 61, 11: 67, 12: 69, 13: 74,
}
UNIQUE_YEARS = sorted(LABEL_TO_YEARS.values())

# Search range for global trajectory modules.
K_MIN = 8
K_MAX = 16

# GO annotation uses Enrichr through GSEApy. It requires network access.
RUN_GO_ENRICHMENT = True
ENRICH_DB = "GO_Biological_Process_2023"
GO_PVALUE_CUTOFF = 0.05
GO_TOP_N_TERMS = 10
API_SLEEP_INTERVAL = 1.0

# Low expression and timepoint filters.
# Set to 0.0 to keep every selected aging gene that is present in the human header.
# A positive threshold can silently remove entire tissues when their selected genes are sparse.
MIN_EXPR_THRESHOLD = 0.0
MIN_TIMEPOINTS_REQUIRED = 2

TISSUE_COLORS = {
    "bladder": "#1f77b4",
    "blood": "#d62728",
    "bone_marrow": "#e377c2",
    "eye": "#17becf",
    "fat": "#ff7f0e",
    "heart": "#2ca02c",
    "kidney": "#9467bd",
    "large_intestine": "#8c564b",
    "liver": "#bcbd22",
    "lung": "#7f7f7f",
    "lymph_node": "#aec7e8",
    "mammary": "#c49c94",
    "muscle": "#98df8a",
    "pancreas": "#f7b6d2",
    "prostate": "#ffbb78",
    "salivary_gland": "#c5b0d5",
    "skin": "#ff9896",
    "small_intestine": "#9edae5",
    "spleen": "#dbdb8d",
    "thymus": "#b5bd61",
    "tongue": "#6b6ecf",
    "trachea": "#6baed6",
    "uterus": "#f781bf",
    "vasculature": "#a65628",
}

os.makedirs(OUTPUT_PLOTS_DIR, exist_ok=True)

logger = logging.getLogger("human_global_trajectory")
logger.setLevel(logging.INFO)
logger.handlers.clear()
logger.addHandler(logging.StreamHandler())
logger.addHandler(logging.FileHandler(os.path.join(OUTPUT_PLOTS_DIR, "run.log"), encoding="utf-8"))
for handler in logger.handlers:
    handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))

if RUN_GO_ENRICHMENT and gp is None:
    raise ImportError("RUN_GO_ENRICHMENT=True, but gseapy is not installed in this notebook kernel. Install gseapy or set RUN_GO_ENRICHMENT=False.")

# ==================== 2. Plot style ====================
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.unicode_minus": False,
    "font.size": 7,
    "axes.labelsize": 7,
    "ytick.labelsize": 7,
    "xtick.labelsize": 7,
    "legend.fontsize": 6.5,
    "axes.linewidth": 0.5,
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "xtick.major.size": 2.0,
    "ytick.major.size": 2.0,
})


def normalize_human_gene(name):
    return str(name).strip().upper()


def parse_human_gene_file(filename):
    """Return tissue_name and hdf5 stem from type_N_tissue_Knee_X_Genes.txt."""
    stem = filename[:-4] if filename.endswith(".txt") else filename
    match = re.match(r"(type_\d+_(.+?))_Knee_\d+_Genes$", stem, flags=re.IGNORECASE)
    if match:
        return match.group(2).lower(), match.group(1).lower()

    raw_name = stem.split("_Knee_")[0] if "_Knee_" in stem else stem.split("_Final_")[0]
    raw_name = re.sub(r"^type_\d+_", "", raw_name, flags=re.IGNORECASE)
    tissue_name = raw_name.lower()
    return tissue_name, tissue_name


def find_hdf5_path(tissue_name, hdf5_stem):
    candidates = [
        os.path.join(DATA_DIR, f"{hdf5_stem}.hdf5"),
        os.path.join(DATA_DIR, f"{hdf5_stem}.h5"),
        os.path.join(DATA_DIR, f"{tissue_name}.hdf5"),
        os.path.join(DATA_DIR, f"{tissue_name}.h5"),
    ]
    for path in candidates:
        if os.path.exists(path):
            return path

    if not os.path.exists(DATA_DIR):
        return None
    for fname in os.listdir(DATA_DIR):
        low = fname.lower()
        if low.endswith((".hdf5", ".h5")) and (hdf5_stem in low or tissue_name in low):
            return os.path.join(DATA_DIR, fname)
    return None


def smooth_trajectory(x, y, x_dense, frac=0.45):
    try:
        y_smoothed = lowess(y, x, frac=frac, return_sorted=False)
        return PchipInterpolator(x, y_smoothed)(x_dense)
    except Exception:
        return np.interp(x_dense, x, y)


def format_go_term_wrapped(term, width=25):
    clean_term = term.split(" (GO:")[0]
    if not clean_term:
        return ""
    return textwrap.fill(clean_term[0].upper() + clean_term[1:], width=width)


def run_enrichr_with_retry(gene_list, module_label):
    if not RUN_GO_ENRICHMENT or gp is None or len(gene_list) < 5:
        return "", pd.DataFrame()

    for attempt in range(1, 4):
        try:
            enr = gp.enrichr(
                gene_list=gene_list,
                gene_sets=ENRICH_DB,
                organism="human",
                outdir=None,
                no_plot=True,
            )
            sig_res = enr.results[enr.results["P-value"] < GO_PVALUE_CUTOFF].copy()
            time.sleep(API_SLEEP_INTERVAL)
            if sig_res.empty:
                return "", pd.DataFrame()

            sig_res = sig_res.sort_values("P-value").head(GO_TOP_N_TERMS)
            sig_res.insert(0, "Module", module_label)
            sig_res.insert(1, "Gene_Count_In_Module", len(gene_list))
            top_term = sig_res.iloc[0]["Term"]
            return format_go_term_wrapped(top_term), sig_res
        except Exception as exc:
            logger.warning("Enrichr attempt %s failed for %s: %s", attempt, module_label, exc)
            time.sleep(2.0)
    return "", pd.DataFrame()


def load_gene_index():
    with open(GLOBAL_GENE_NAMES_FILE, "r", encoding="utf-8") as handle:
        return {normalize_human_gene(line): i for i, line in enumerate(handle) if line.strip()}


def load_selected_genes(path):
    with open(path, "r", encoding="utf-8") as handle:
        return [normalize_human_gene(line) for line in handle if line.strip()]


def build_tissue_z_matrices(gene_to_idx):
    gene_files = sorted(f for f in os.listdir(INPUT_GENE_DIR) if f.endswith(".txt"))
    multi_tissue_z_data = {}
    actual_tissues_found = []

    logger.info("Step 1: Build per-tissue human aging-gene Z-score matrices.")
    for gene_file in tqdm(gene_files, desc="Loading human tissues"):
        tissue_name, hdf5_stem = parse_human_gene_file(gene_file)
        gene_path = os.path.join(INPUT_GENE_DIR, gene_file)
        selected_genes = load_selected_genes(gene_path)
        valid_genes = [gene for gene in selected_genes if gene in gene_to_idx]
        if not valid_genes:
            logger.warning("[%s] no selected genes found in header; skipped.", tissue_name)
            continue

        hdf5_path = find_hdf5_path(tissue_name, hdf5_stem)
        if hdf5_path is None:
            logger.warning("[%s] hdf5 file not found; skipped.", tissue_name)
            continue

        with h5py.File(hdf5_path, "r") as handle:
            sorted_pairs = sorted((gene_to_idx[gene], gene) for gene in valid_genes)
            indices = [idx for idx, _ in sorted_pairs]
            plot_genes = [gene for _, gene in sorted_pairs]
            raw_data = handle["data"][:, indices]
            labels = handle["label"][:, 0]

        df_raw = pd.DataFrame(raw_data, columns=plot_genes)
        df_raw["Age_Years"] = pd.Series(labels).map(LABEL_TO_YEARS)
        df_raw = df_raw.dropna(subset=["Age_Years"])
        if df_raw["Age_Years"].nunique() < MIN_TIMEPOINTS_REQUIRED:
            logger.warning("[%s] fewer than %s age points; skipped.", tissue_name, MIN_TIMEPOINTS_REQUIRED)
            continue

        if MIN_EXPR_THRESHOLD and MIN_EXPR_THRESHOLD > 0:
            gene_means = df_raw.drop(columns=["Age_Years"]).mean()
            high_expr_genes = gene_means[gene_means > MIN_EXPR_THRESHOLD].index.tolist()
            if not high_expr_genes:
                logger.warning("[%s] no genes passed expression filter; skipped.", tissue_name)
                continue
        else:
            high_expr_genes = plot_genes

        df_log = np.log1p(df_raw[high_expr_genes].astype(float))
        df_log["Age_Years"] = df_raw["Age_Years"].values
        grouped_mean = df_log.groupby("Age_Years").mean().reindex(UNIQUE_YEARS)
        grouped_mean = grouped_mean.interpolate(method="linear").bfill().ffill()

        z_scores = StandardScaler().fit_transform(grouped_mean)
        multi_tissue_z_data[tissue_name] = pd.DataFrame(
            z_scores,
            index=grouped_mean.index,
            columns=grouped_mean.columns,
        )
        actual_tissues_found.append(tissue_name)
        logger.info("[%s] retained genes: %s", tissue_name, len(high_expr_genes))

    return multi_tissue_z_data, actual_tissues_found


def build_global_z_matrix(multi_tissue_z_data):
    logger.info("Step 2: Build global consensus Z-score matrix.")
    if not multi_tissue_z_data:
        raise ValueError("No valid tissue Z-score matrices were built.")

    all_genes_union = sorted(set().union(*(df.columns for df in multi_tissue_z_data.values())))
    global_sum = pd.DataFrame(0.0, index=UNIQUE_YEARS, columns=all_genes_union)
    gene_counts = pd.Series(0, index=all_genes_union, dtype=float)

    for z_df in multi_tissue_z_data.values():
        global_sum.loc[:, z_df.columns] = global_sum.loc[:, z_df.columns].add(z_df, fill_value=0.0)
        gene_counts.loc[z_df.columns] += 1

    global_z_matrix = global_sum.div(gene_counts, axis=1).dropna(axis=1)
    global_z_matrix.to_csv(os.path.join(OUTPUT_PLOTS_DIR, "Human_Global_Z_Matrix.csv"))
    return global_z_matrix


def choose_best_k(global_z_matrix):
    logger.info("Step 3: Evaluate global module number K.")
    x = global_z_matrix.T.values
    if x.shape[0] < 3:
        raise ValueError("Too few genes for clustering.")

    k_min = min(K_MIN, x.shape[0] - 1)
    k_max = min(K_MAX, x.shape[0] - 1)
    if k_min > k_max:
        k_min = k_max

    best_k = k_min
    best_sil_score = -1.0
    sample_size = min(2000, x.shape[0])

    for k in tqdm(range(k_min, k_max + 1), desc="Evaluating K"):
        labels = AgglomerativeClustering(n_clusters=k, linkage="ward").fit_predict(x)
        scores = []
        for seed in [42, 123, 2024]:
            scores.append(silhouette_score(x, labels, sample_size=sample_size, random_state=seed))
        sil_score = float(np.mean(scores))
        if sil_score > best_sil_score:
            best_sil_score = sil_score
            best_k = k

    logger.info("Best K = %s, silhouette = %.4f", best_k, best_sil_score)
    return best_k


def cluster_global_modules(global_z_matrix, best_k):
    x = global_z_matrix.T.values
    labels = AgglomerativeClustering(n_clusters=best_k, linkage="ward").fit_predict(x)
    modules_dict = {
        f"Module {i + 1}": global_z_matrix.columns[labels == i].tolist()
        for i in range(best_k)
    }

    module_rows = []
    for module_name, genes in modules_dict.items():
        for gene in genes:
            module_rows.append({"Module": module_name, "Gene": gene})
    pd.DataFrame(module_rows).to_csv(
        os.path.join(OUTPUT_PLOTS_DIR, "Human_Global_Module_Assignments.csv"),
        index=False,
    )

    module_deltas = {}
    for module_name, genes in modules_dict.items():
        traj_mean = global_z_matrix[genes].mean(axis=1)
        module_deltas[module_name] = traj_mean.iloc[-1] - traj_mean.iloc[0]
    sorted_module_keys = sorted(modules_dict, key=lambda key: module_deltas[key])
    return modules_dict, sorted_module_keys


def plot_global_modules(
    global_z_matrix,
    multi_tissue_z_data,
    actual_tissues_found,
    modules_dict,
    sorted_module_keys,
):
    logger.info("Step 4: Plot human global gene trajectory modules.")
    cols = 4
    rows = math.ceil(len(sorted_module_keys) / cols)
    fig_w = 178 / 25.4
    fig_h = (rows * 45 + 65) / 25.4
    fig, axes = plt.subplots(rows, cols, figsize=(fig_w, fig_h), sharex=False, sharey=False)
    axes = np.array(axes).flatten()

    for i in range(len(sorted_module_keys), rows * cols):
        fig.delaxes(axes[i])

    x_smooth = np.linspace(min(UNIQUE_YEARS), max(UNIQUE_YEARS), 300)
    pe_white_border = [path_effects.Stroke(linewidth=2.5, foreground="white"), path_effects.Normal()]
    pe_white_border_mean = [path_effects.Stroke(linewidth=1.8, foreground="white"), path_effects.Normal()]

    module_summary_rows = []
    all_go_results = []

    for idx, module_key in enumerate(tqdm(sorted_module_keys, desc="Plotting modules")):
        ax = axes[idx]
        module_genes = modules_dict[module_key]
        module_label = f"Module {idx + 1}"
        go_text, go_df = run_enrichr_with_retry(module_genes, module_label)
        if not go_df.empty:
            go_df.insert(1, "Original_Module", module_key)
            all_go_results.append(go_df)
        ax.set_box_aspect(1)

        tissue_trajs = {}
        for tissue in actual_tissues_found:
            tissue_df = multi_tissue_z_data.get(tissue)
            if tissue_df is None:
                continue
            valid_genes = [gene for gene in module_genes if gene in tissue_df.columns]
            if valid_genes:
                tissue_trajs[tissue] = tissue_df[valid_genes].mean(axis=1).values

        if not tissue_trajs:
            continue

        all_trajs_matrix = np.array(list(tissue_trajs.values()))
        consensus_traj = np.mean(all_trajs_matrix, axis=0)
        consensus_std = np.std(all_trajs_matrix, axis=0)

        correlations = {}
        for tissue, traj in tissue_trajs.items():
            other_trajs = [values for key, values in tissue_trajs.items() if key != tissue]
            if not other_trajs:
                correlations[tissue] = 1.0
                continue
            consensus_excluding_tissue = np.mean(other_trajs, axis=0)
            if np.std(traj) > 0 and np.std(consensus_excluding_tissue) > 0:
                correlations[tissue] = pearsonr(traj, consensus_excluding_tissue)[0]
            else:
                correlations[tissue] = 0.0

        top_tissues = sorted(correlations, key=correlations.get, reverse=True)[:3]
        module_summary_rows.append({
            "Module": f"Module {idx + 1}",
            "Original_Module": module_key,
            "Gene_Count": len(module_genes),
            "Delta_Last_First": float(consensus_traj[-1] - consensus_traj[0]),
            "Top_Tissues": ";".join(top_tissues),
            "GO_Term": go_text.replace("\n", " "),
        })

        for tissue, traj in tissue_trajs.items():
            if tissue not in top_tissues:
                y_smooth = smooth_trajectory(UNIQUE_YEARS, traj, x_smooth, frac=0.45)
                ax.plot(x_smooth, y_smooth, color="#d3d3d3", alpha=0.5, linewidth=0.6, zorder=1)

        cons_smooth = smooth_trajectory(UNIQUE_YEARS, consensus_traj, x_smooth, frac=0.5)
        std_smooth = smooth_trajectory(UNIQUE_YEARS, consensus_std, x_smooth, frac=0.5)
        ax.fill_between(
            x_smooth,
            cons_smooth - std_smooth,
            cons_smooth + std_smooth,
            color="black",
            alpha=0.1,
            zorder=2,
            linewidth=0,
        )

        end_positions = []
        for tissue in top_tissues:
            y_smooth = smooth_trajectory(UNIQUE_YEARS, tissue_trajs[tissue], x_smooth, frac=0.45)
            end_positions.append({"tissue": tissue, "y": y_smooth[-1], "y_smooth": y_smooth})

        end_positions.sort(key=lambda item: item["y"])
        for pos_idx in range(1, len(end_positions)):
            if end_positions[pos_idx]["y"] - end_positions[pos_idx - 1]["y"] < 0.35:
                end_positions[pos_idx]["y"] = end_positions[pos_idx - 1]["y"] + 0.35

        for item in end_positions:
            tissue = item["tissue"]
            color = TISSUE_COLORS.get(tissue, "#333333")
            ax.plot(
                x_smooth,
                item["y_smooth"],
                color=color,
                alpha=0.95,
                linewidth=1.2,
                zorder=3,
                path_effects=pe_white_border,
            )
            ax.text(
                max(UNIQUE_YEARS) + 1.5,
                item["y"],
                tissue.replace("_", " ").title(),
                color=color,
                fontsize=5.5,
                fontname="Arial",
                fontweight="bold",
                va="center",
                zorder=5,
            )

        ax.plot(
            x_smooth,
            cons_smooth,
            color="black",
            alpha=0.8,
            linewidth=1.0,
            linestyle="--",
            dashes=(4, 2),
            zorder=4,
            path_effects=pe_white_border_mean,
        )

        title = f"Module {idx + 1} (n={len(module_genes)})"
        if go_text:
            num_lines = go_text.count("\n") + 1
            ax.set_title(title, fontsize=7, fontweight="bold", color="#333333", pad=8 + num_lines * 8)
            ax.text(
                0.5,
                1.02,
                go_text,
                transform=ax.transAxes,
                ha="center",
                va="bottom",
                fontsize=6.5,
                color="#555555",
                linespacing=1.2,
            )
        else:
            ax.set_title(title, fontsize=7, fontweight="bold", color="#333333", pad=4)

        ax.set_xticks([0, 20, 40, 60, 80])
        ax.set_xticklabels(["0", "20", "40", "60", "80"])
        ax.set_yticks([-2, -1, 0, 1, 2])
        ax.set_yticklabels(["-2", "-1", "0", "1", "2"])
        ax.set_xlim(0, max(UNIQUE_YEARS) + 18)
        ax.set_ylim(-2.8, 2.8)
        ax.axhline(0, color="black", linestyle="-", linewidth=0.4, alpha=0.3, zorder=0)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    pd.DataFrame(module_summary_rows).to_csv(
        os.path.join(OUTPUT_PLOTS_DIR, "Human_Global_Module_Summary.csv"),
        index=False,
    )

    go_out = os.path.join(OUTPUT_PLOTS_DIR, "Human_Global_Module_GO_Enrichment_TopTerms.csv")
    if all_go_results:
        pd.concat(all_go_results, ignore_index=True).to_csv(go_out, index=False)
    else:
        pd.DataFrame(columns=["Module", "Original_Module", "Gene_Count_In_Module", "Term", "P-value"]).to_csv(go_out, index=False)
    logger.info("Saved GO enrichment table: %s", go_out)

    patch_handles = []
    for tissue in sorted(actual_tissues_found):
        color = TISSUE_COLORS.get(tissue, "#333333")
        patch_handles.append(mpatches.Patch(color=color, label=tissue.replace("_", " ").title()))

    if patch_handles:
        fig.legend(
            handles=patch_handles,
            loc="upper center",
            bbox_to_anchor=(0.5, 0.975),
            ncol=8,
            frameon=False,
            columnspacing=1.0,
            handlelength=0.7,
            handleheight=0.7,
        )

    fig.text(0.5, 0.01, "Age (years)", ha="center", va="center", fontsize=7, fontweight="bold")
    fig.text(0.01, 0.5, "Scaled expression (Z-score)", ha="center", va="center", rotation="vertical", fontsize=7, fontweight="bold")
    fig.subplots_adjust(top=0.82, bottom=0.09, left=0.08, right=0.90, hspace=0.65, wspace=0.5)

    out_pdf = os.path.join(OUTPUT_PLOTS_DIR, "Human_Global_Gene_Trajectory_Modules.pdf")
    with PdfPages(out_pdf) as pdf:
        pdf.savefig(fig, transparent=True)
    plt.close(fig)
    logger.info("Saved figure: %s", out_pdf)


def main():
    gene_to_idx = load_gene_index()
    multi_tissue_z_data, actual_tissues_found = build_tissue_z_matrices(gene_to_idx)
    global_z_matrix = build_global_z_matrix(multi_tissue_z_data)
    best_k = choose_best_k(global_z_matrix)
    modules_dict, sorted_module_keys = cluster_global_modules(global_z_matrix, best_k)
    plot_global_modules(
        global_z_matrix=global_z_matrix,
        multi_tissue_z_data=multi_tissue_z_data,
        actual_tissues_found=actual_tissues_found,
        modules_dict=modules_dict,
        sorted_module_keys=sorted_module_keys,
    )
    logger.info("Done. Outputs are in: %s", OUTPUT_PLOTS_DIR)


if __name__ == "__main__":
    main()
